# Kuznetsov et al. 1994: Recreation
Bulletin of Mathematical Biology, Vol. 56, No. 2, pp. 295-321

Nonlinear Dynamics of Immunogenic Tumors: Parameter Estimation and Global Bifurcation Analysis


## 1. Basic Model

EC (effector cells) and TC (tumor cells). The basic interaction can be described as:

$E$ + $T$ --> C by $k_{-1}$

$E$ + $T$ <-- C by $k_{-1}$

and

C --> $E$ + $T^*$ by $k_{2}$

C --> $E^*$ + $T$ by $k_{3}$


With E = local concentration of **effector cells**, T = local concentration of **tumor cells**, C = **effector cell-tumor cell conjugates**, $E^*$ = **inactivated effector cells**, and $T^*$ = "lethally hit" TCs.

## 2. Full Model

Effector cells (E) and tumor cells (T) interact via short-lived conjugates (C). The full 5-equation system tracks unbound effector cells, unbound tumor cells, EC-TC conjugates, inactivated effectors, and lethally-hit tumor cells:

$\frac{dE}{dt} = s + F(C,T) - d_1 E - k_1 ET + (k_{-1} + k_2)C$

$\frac{dT}{dt} = aT(1 - bT_\text{tot}) - k_1 ET + (k_{-1} + k_3)C$

$\frac{dC}{dt} = k_1 ET - (k_{-1} + k_2 + k_3)C$

$\frac{dE^*}{dt} = k_3 C - d_2 E^*$

$\frac{dT^*}{dt} = k_2 C - d_3 T^*$

The stimulated effector recruitment term is:

$F(C,T) = \frac{fC}{g + T}$

Since $E^*$ and $T^*$ do not feed back into the other equations, equations (4) and (5) are **slaves** and are dropped. The remaining system is equations (1)-(3).


## 3. Quasi-Steady-State Reduction

Conjugate formation and dissociation occur on a timescale of minutes to hours; effector cell proliferation and influx occur on a timescale of days. This separation of timescales justifies setting $dC/dt \approx 0$, giving:

$$C \approx KET, \qquad K = \frac{k_1}{k_2 + k_3 + k_{-1}}$$

Conjugates also comprise only 1-10% of total cells, so $T_\text{tot} = T + C \approx T$. Substituting both approximations and absorbing constants into new parameters ($p = fK$, $m = Kk_3$, $n = Kk_2$, $d = d_1$) collapses the 3-equation system to **2 equations**:

$$\frac{dE}{dt} = s + \frac{pET}{g+T} - mET - dE$$

$$\frac{dT}{dt} = aT(1-bT) - nET$$


## 4. Nondimensionalization

Scaling $x = E/E_0$, $y = T/T_0$ with $E_0 = T_0 = 10^6$ cells, and $\tau = nT_0 t$:

$\frac{dx}{d\tau} = \sigma + \frac{\rho x y}{\eta + y} - \mu xy - \delta x$

$\frac{dy}{d\tau} = \alpha y(1 - \beta y) - xy$

The seven nondimensional parameters are:

| Symbol | Definition (in terms of original vars) |
|--------|-----------------------------------------|
| $\sigma$ | $s / (nE_0 T_0)$ |
| $\rho$ | $p / (nT_0)$ |
| $\eta$ | $g / T_0$ |
| $\mu$ | $m/n = k_3/k_2$ |
| $\delta$ | $d / (nT_0)$ |
| $\alpha$ | $a / (nT_0)$ |
| $\beta$ | $bT_0$ |

Values are provided lower in the notebook. $\mu = k_3/k_2$ is the ratio of effector inactivation rate to tumor kill rate, and turns out to be the critical bifurcation parameter governing whether sneaking-through is possible.

## Setup

This notebook relies on three local modules that live alongside it:

- `kuznetsov_model.py`: dimensional & nondimensional model, parameters, experimental data
- `phase_portrait.py`: the `PhasePortraitPlotter` class (Figures 3, 5, 6, 8, 9)
- `fig4_helpers.py`: region-classification helpers for the Figure 4 bifurcation diagram

Running the cell below pulls everything into scope. All figure cells then run unchanged.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import fsolve, brentq
from IPython.display import display, Image

from kuznetsov_model import (params, nd_params, sigma, rho, eta, mu,
                             delta, alpha, beta, times, log_values, values,
                             model, model_nd, E0_nd, T0_nd)
from phase_portrait import PhasePortraitPlotter
from fig4_helpers import classify_local, in_region_5

print("Setup loaded. Nondimensional parameters:")
for k, v in nd_params.items():
    print(f"  {k:6s} = {v:.5g}")


## Experimental Data

Digitized from Figure 1 of the paper using WebPlotDigitizer (https://apps.automeris.io/wpd/).
This is BCL1 tumor growth in chimeric mice, curve 2 (5e5 initial tumor cells).

In [ ]:
# Experimental data (times, log_values, values) is imported from kuznetsov_model.
print("Time (days):", times)
print("Tumor cells:", values)

plt.plot(times, log_values, 'o-')
plt.xlabel('Time (days)')
plt.ylabel(r'log$_{10}$(tumor cells)')
plt.title('Digitized BCL$_1$ growth data (Siu et al. 1986, curve 2)')
plt.show()


## The Model (Equations 4a, 4b)

After quasi-steady-state approximation on conjugates C (fast timescale ~hours vs slow proliferation ~days),
and approximating T_tot ≈ T, the full 5-equation system reduces to:

$\frac{dE}{dt} = s + \frac{pET}{g+T} - mET - dE$

$\frac{dT}{dt} = aT(1-bT) - nET$

E = effector cells (CTL/NK), T = tumor cells

Term by term:
- s          : baseline influx of effector cells
- pET/(g+T)  : tumor-stimulated effector recruitment (saturates at large T)
- mET        : effector inactivation by tumor
- dE         : effector death/emigration
- aT(1-bT)   : logistic tumor growth, carrying capacity = 1/b
- nET        : tumor killing by effectors

## Parameters (Section 3)

a, b estimated from logistic fit to immune-free (non-chimeric) tumor growth.
s estimated from steady-state condition: s ≈ E_p * d, where E_p = 3.2e5 CTL precursors in spleen.
p, g, m, n, d estimated by nonlinear least squares to fit chimeric mouse data.

## Figure 1: Experimental Data

Initial conditions: E(0) = 3.2e5 (CTL precursor count), T(0) = 5e5 (tumor dose for curve 2).
Plot log10(T) vs time and overlay the digitized data points.

In [ ]:
# Initial conditions
E0 = 3.2e5
T0 = 5e5  # curve 2 from paper

# TODO: run solve_ivp and plot vs experimental data
sol = solve_ivp(model, (0,90), [E0, T0], args=(params['s'], params['p'], params['g'],
                                                 params['m'], params['d'], params['a'],
                                                 params['b'], params['n']))

plt.plot(sol.t, np.log10(sol.y[1]), label='model')
plt.scatter(times, log_values, label='data', color='red')
plt.xlabel('Time (days)')
plt.ylabel('log$_{10}$T (tumor cells)')
plt.title('Figure 1a: BCL$_1$ Tumor Growth in Chimeric Mice')
plt.legend(['Model (3.2×10⁵ initial effectors)', 'Data (Siu et al. 1986, curve 2)'])
plt.xlim(0, 90)
plt.show()



## Non-Dimensionalization (Section 4)

Scale: x = E/E0, y = T/T0 with E0 = T0 = 1e6 cells.
Rescale time: τ = n*T0*t.


$\frac{dx}{d\tau} = \sigma + \frac{\rho x y}{\eta + y} - \mu xy - \delta x$

$\frac{dy}{d\tau} = \alpha y(1 - \beta y) - xy$

The seven nondimensional parameters are:

| Symbol | Definition (in terms of original vars) |
|--------|-----------------------------------------|
| $\sigma$ | $s / (nE_0 T_0)$ |
| $\rho$ | $p / (nT_0)$ |
| $\eta$ | $g / T_0$ |
| $\mu$ | $m/n = k_3/k_2$ |
| $\delta$ | $d / (nT_0)$ |
| $\alpha$ | $a / (nT_0)$ |
| $\beta$ | $bT_0$ |

## Nullclines and Steady States (Section 5)

x-nullcline (dx/dτ = 0):  x = f(y) = σ / (δ + μy - ρy/(η+y))
y-nullclines (dy/dτ = 0): y = 0  OR  x = g(y) = α(1 - βy)

Setting f(y) = g(y) gives a cubic in y:
    C3*y^3 + C2*y^2 + C1*y + C0 = 0

where:
  $C_0 = η*(σ/α - δ)$

  $C_1 = σ/α + ρ - μη - δ + δηβ$

  $C_2 = -μ + (μη + δ - ρ)*β$

  $C_3 = μβ$

At estimated parameters there are 4 steady states: A (tumor-free saddle),
B (dormancy, stable), C (saddle, separatrix), D (escape, stable).

In [ ]:
# Plot nullclines f(y) and g(y)
def f(y):
  # x-nullcline
  return nd_params["sigma"] / (delta + mu * y - rho * y / (eta + y))

def g(y):
  # y-nullcline (nontrivial)
  return alpha * (1 - beta * y)

def cubic(y):
    # solution to f(y) - g(y) = 0
    return f(y) - g(y)


y_plot = np.linspace(0, 800, 1000)

# DEBUGGING: mask out points where denominator is near zero
threshold = 0.01  # tunable
denom = delta + mu*y_plot - rho*y_plot/(eta + y_plot)
mask = np.abs(denom) > threshold
f_vals = np.where(mask, sigma / denom, np.nan)  # nan breaks the line
g_vals = alpha * (1 - beta * y_plot)
# DEBUGGING: end block

plt.plot(f_vals, y_plot, label='x-nullcline f(y)')  # x horizontal, y vertical
plt.plot(g_vals, y_plot, label='y-nullcline g(y)')
# mark B, C, D intersections
for (x_s, y_s, name) in [(1.608, 8.186,'B'), (0.759, 268.023,'C'), (0.173, 447.026,'D')]:
    plt.plot(x_s, y_s, 'ko', ms=5)
    plt.annotate(name, (x_s, y_s), textcoords='offset points', xytext=(5,3))

plt.plot(sigma / delta, 0,'ko', ms=5)
plt.annotate("A", (sigma / delta, 0), textcoords='offset points', xytext=(-10,4))

## PLOT STYLING

plt.title('Nullclines at Estimated Parameters\n'
          r'x-nullcline: $\dot{x}=0$,  y-nullcline: $\dot{y}=0$')
plt.xlabel('x (nondimensional effector)')
plt.ylabel('y (nondimensional tumor)')
plt.xlim(0, 4)
plt.ylim(0, 500)
plt.legend()
plt.show()

## Figure 2: Qualitative forms of f(y)

In [ ]:
# Figure 2: Qualitative forms of f(y)
#
# All panels share: η=2, μ=0.5, δ=0.5
#   √(ημ)=1, √δ≈0.707
#   upper threshold (√(ημ)+√δ)² ≈ 2.914
#   lower threshold (√(ημ)-√δ)² ≈ 0.086
#   ημ = 1.0
#
# Panel (a): ρ=4.0  ≥ 2.914         → two POSITIVE asymptotes ȳ₁≈4.56, ȳ₂≈0.44
#            σ=0.5 so f(0)=σ/δ=1.0 and A=(1,0) sits at a visible x value
#            β=0.1  so g=0 at y=10, keeping upper-branch crossings in window
#            g_alphas=[0.7,1.4,2.2] → verified 0,1,3 crossings in x>0,y>0
#
# Panel (b): ρ=1.8, 0.086<1.8<2.914 and 1.8>ημ=1.0
#            → no asymptotes; local max of f(y) at y≈0.68>0
#            No g-lines (paper shows f(y) shape only)
#
# Panel (c): ρ=0.6, 0.086<0.6<2.914 and 0.6<ημ=1.0
#            → no asymptotes; both extrema of f(y) at y<0 (S-shape straddles x-axis)
#            No g-lines
#
# Panel (d): ρ=0.01 ≤ 0.086
#            → two NEGATIVE asymptotes ȳ₁≈-1.02, ȳ₂≈-1.96
#            f(y) monotone decreasing for y>0; S-shape entirely below x-axis
#            No g-lines

eta_demo   = 2.0
mu_demo    = 0.5
delta_demo = 0.5

# σ: panel (a) uses 0.5 so A=(σ/δ,0)=(1,0) is at a visible x; others use 0.3
sigmas       = [0.5,  0.3,  0.3,  0.3]
panel_rhos   = [4.0,  1.8,  0.6,  0.01]
panel_labels = ['(a)', '(b)', '(c)', '(d)']

# y arrays: (a)/(b) reach y=10 to capture upper-branch crossings; (c)/(d) symmetric
y_ab = np.linspace(-2.0, 10.0, 8000)
y_cd = np.linspace(-4.0,  3.0, 8000)
y_arrays = [y_ab, y_ab, y_cd, y_cd]

# Axis limits
x_lims = [(-1.5, 4.0), (-0.5, 2.0), (-0.5, 2.0), (-0.5, 2.0)]
y_lims = [(-2.0, 10.0), (-2.0, 10.0), (-4.0, 3.0), (-4.0, 3.0)]

# g-lines in panel (a) only; β=0.1 keeps g>0 up to y=10
beta_demo = 0.1
g_alphas_per_panel = [
    [0.7, 1.4, 2.2],  # (a): verified 0, 1, 3 crossings in x>0, y>0
    [],               # (b): no g-lines
    [],               # (c): no g-lines
    [],               # (d): no g-lines
]
g_line_labels = [['(i)', '(ii)', '(iii)'], [], [], []]
g_styles      = ['k--', 'k-.', 'k:']

fig2, axes = plt.subplots(2, 2, figsize=(11, 10))
axes = axes.flatten()

for ax, sigma_demo, rho_demo, panel_label, y, xlim, ylim, g_als, g_llbls in zip(
        axes, sigmas, panel_rhos, panel_labels,
        y_arrays, x_lims, y_lims,
        g_alphas_per_panel, g_line_labels):

    # f(y) = σ / (δ + μy - ρy/(η+y))
    denom  = delta_demo + mu_demo*y - rho_demo*y / (eta_demo + y)
    mask   = np.abs(denom) > 0.015
    f_vals = np.where(mask, sigma_demo / denom, np.nan)
    # clip to x window only; negative x is allowed (curve lives there)
    f_vals = np.where((f_vals > xlim[0]) & (f_vals < xlim[1]), f_vals, np.nan)

    ax.plot(f_vals, y, 'k-', lw=2)

    # asymptotes: roots of μy² + (δ+μη-ρ)y + δη = 0
    A_c  = mu_demo
    B_c  = delta_demo + mu_demo*eta_demo - rho_demo
    C_c  = delta_demo * eta_demo
    disc = B_c**2 - 4*A_c*C_c
    if disc >= 0:
        roots = sorted([
            (-B_c + np.sqrt(disc)) / (2*A_c),
            (-B_c - np.sqrt(disc)) / (2*A_c),
        ], reverse=True)
        lbl_map = {0: r'$\bar{y}_1$', 1: r'$\bar{y}_2$'}
        for k, ybar in enumerate(roots):
            if ylim[0] < ybar < ylim[1]:
                ax.axhline(ybar, color='k', lw=0.9, ls='--')
                ax.text(xlim[1] * 0.96, ybar + 0.015*(ylim[1]-ylim[0]),
                        lbl_map[k], fontsize=9, ha='right', va='bottom')

    # g(y) lines and intersection dots, panel (a) only
    for g_a, sty, llbl in zip(g_als, g_styles, g_llbls):
        g_vals = g_a * (1 - beta_demo*y)
        # strict first-quadrant clip: x>0 AND y>0
        g_plot = np.where((g_vals > 0) & (y > 0) & (g_vals < xlim[1]), g_vals, np.nan)
        ax.plot(g_plot, y, sty, lw=1.5)

        # label near the top of each visible g-line
        if llbl:
            valid = np.where(np.isfinite(g_plot))[0]
            if len(valid):
                idx = valid[-1]
                ax.text(g_plot[idx] + 0.04*(xlim[1]-xlim[0]),
                        y[idx], llbl, fontsize=9, va='center')

        # intersection dots: f=g strictly in first quadrant
        diff = np.where(
            mask & (f_vals > 0) & (f_vals < xlim[1]) & (g_vals > 0) & (y > 0),
            f_vals - g_vals, np.nan)
        valid_idx = np.where(np.isfinite(diff))[0]
        if len(valid_idx) >= 2:
            signs   = np.sign(diff[valid_idx])
            changes = np.where(np.diff(signs) != 0)[0]
            for ci in changes:
                i0, i1 = valid_idx[ci], valid_idx[ci + 1]
                d0, d1 = diff[i0], diff[i1]
                y_cross = y[i0] - d0 * (y[i1] - y[i0]) / (d1 - d0)
                x_cross = g_a * (1 - beta_demo * y_cross)
                if x_cross > 0 and 0 < y_cross < ylim[1]:
                    ax.plot(x_cross, y_cross, 'ko', ms=6, zorder=5)

    # steady state A = (σ/δ, 0), always present
    x_A = sigma_demo / delta_demo
    ax.plot(x_A, 0, 'ko', ms=6, zorder=5)
    ax.text(x_A + 0.03*(xlim[1]-xlim[0]), -0.03*(ylim[1]-ylim[0]),
            'A', fontsize=9, ha='left', va='top')

    # schematic axes (no ticks, arrows at ends, origin label)
    ax.spines[['top', 'right', 'bottom', 'left']].set_visible(False)
    ax.set_xticks([]); ax.set_yticks([])
    ax.axhline(0, color='k', lw=0.9)
    ax.axvline(0, color='k', lw=0.9)
    ax.annotate('', xy=(xlim[1], 0), xytext=(xlim[0], 0),
                arrowprops=dict(arrowstyle='->', lw=1.0, color='k'))
    ax.annotate('', xy=(0, ylim[1]), xytext=(0, ylim[0]),
                arrowprops=dict(arrowstyle='->', lw=1.0, color='k'))
    ax.text(xlim[1]*0.98, -0.06*(ylim[1]-ylim[0]), 'x', fontsize=10, ha='right')
    ax.text(0.03*(xlim[1]-xlim[0]), ylim[1]*0.97,  'y', fontsize=10, va='top')
    ax.text(-0.05*(xlim[1]-xlim[0]), -0.03*(ylim[1]-ylim[0]), '0',
            fontsize=8, ha='right', va='top')

    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_title(panel_label, fontsize=13, fontweight='bold', loc='right')

plt.suptitle(r'Figure 2: Qualitative Forms of $f(y)$ [solid line] and $g(y)$ [dashed lines]',
             fontsize=12)
plt.tight_layout()
plt.show()

## Figure 3: Phase Portrait

Reproduce Figure 3 from the paper at the estimated nondimensional parameter values.
Show trajectories from initial conditions (i)-(iv), mark steady states, and draw the separatrix.

In [ ]:
fig3_plotter = PhasePortraitPlotter.from_nd_params(nd_params)

fig, ax = plt.subplots(figsize=(7, 6))
ax.set_xlim(0, 4)
ax.set_ylim(-10, 500)

fig3_plotter.draw_streamplot(ax, x_max=4, y_max=500)
fig3_plotter.plot_saddle_manifolds(ax, eps=0.01, T_stable=100, T_unstable=100)
fig3_plotter.plot_basin_boundary(ax, np.linspace(0.1, 1.13, 40))

trajectories = [
    (0.25, 50,   'b', '(i)'),
    (0.5, 100,   'r', '(ii)'),
    (1.5, 450,   'g', '(iii)'),
    (0.5, 450,   'm', '(iv)'),
]
t_eval = np.linspace(0, 300, 1000)
for x0, y0, color, label in trajectories:
    sol = solve_ivp(fig3_plotter.ode, (0, 300), [x0, y0],
                    t_eval=t_eval, max_step=1.0, rtol=1e-8, atol=1e-10)
    ax.plot(sol.y[0], sol.y[1], color=color, lw=1.2, label=label)

fig3_plotter.plot_fixed_points(ax)
fig3_plotter.label_fixed_points(ax)

ax.set_xlabel('x (effector)')
ax.set_ylabel('y (tumor)')
ax.legend()
plt.title("Figure 3: Nondimensional Phase Portrait")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.set_xlim(0, 4)
ax.set_ylim(0.1, 500)
ax.set_yscale('log')

trajectories = [
    (0.25, 50,   'b', '(i)'),
    (0.5, 100,   'r', '(ii)'),
    (1.5, 450,   'g', '(iii)'),
    (0.5, 450,   'm', '(iv)'),
    (0.4, 0.05,  '#000000', '(v)'),
]
t_eval = np.linspace(0, 300, 1000)
for x0, y0, color, label in trajectories:
    sol = solve_ivp(fig3_plotter.ode, (0, 300), [x0, y0],
                    t_eval=t_eval, max_step=1.0, rtol=1e-8, atol=1e-10)
    ax.plot(sol.y[0], sol.y[1], color=color, lw=1.2, label=label)

fig3_plotter.plot_fixed_points(ax, log_y=True)
fig3_plotter.label_fixed_points(ax, log_y=True)

ax.set_xlabel('x (effector)')
ax.set_ylabel('y (tumor)')
ax.legend()
plt.title("Figure 3: Nondimensional Phase Portrait (log scale)")
plt.show()

This figure isn't from the paper. It's just an interesting one to include.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.set_xlim(0, 4)
ax.set_ylim(0, 500)

fig3_plotter.draw_streamplot(ax, x_max=4, y_max=500)
fig3_plotter.plot_nullclines(ax, y_max=800)
fig3_plotter.plot_saddle_manifolds(ax, eps=0.01, T_stable=200, T_unstable=200)
fig3_plotter.plot_basin_boundary(ax, np.linspace(0.1, 1.13, 40))

fig3_plotter.plot_fixed_points(ax)
fig3_plotter.label_fixed_points(ax)

ax.set_xlim(0, 4)
ax.set_ylim(0, 500)
ax.set_xlabel('x (nondimensional effector)')
ax.set_ylabel('y (nondimensional tumor)')
ax.set_title('Figure 3b: Phase Portrait with Nullclines\n'
             r'x-nullcline $\dot{x}=0$,  y-nullcline $\dot{y}=0$')
ax.legend()
plt.tight_layout()
plt.show()


## Figure 4: Bifurcation Diagram

Vary δ (effector death rate) and σ (effector source rate) while holding other params fixed.
Map out the 5 dynamical regions.

Transitions:
- 1-2, 4-3 : transcritical bifurcation (steady states A and B)
- 1-4, 2-3 : saddle-node bifurcation (steady states C and D)
- 2-5       : saddle-node bifurcation (steady states C and B)
- 3-5       : heteroclinic connection (A and C), the "vital barrier"
               this is a GLOBAL bifurcation, not detectable from linearization

Estimated parameters sit in Region 3, near the 3↔5 boundary.

In [ ]:
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from scipy.ndimage import gaussian_filter1d

### Can take up to 11-12 minutes

# =============================================================================
# Figure 4: Bifurcation Diagram (δ, σ parameter plane)
# Regions 3 and 5 are now correctly separated via heteroclinic detection.
#
# Strategy:
#   1. classify_local(): counts steady states + eigenvalue signs.
#                          Returns 1, 2, 4, or 3 (where 3 means "3 or 5").
#   2. in_region_5(): for points tagged 3-or-5, shoots forward along A's
#                          unstable manifold. If the trajectory escapes to high
#                          y (→ D), the heteroclinic connection has been crossed
#                          and we are in region 5.  Otherwise region 3.
#   3. Grid scan: single pass; classify_local first, then heteroclinic
#                          check only for the 3/5 subset (~20-30% of grid).
#   4. Boundary curves: extracted from grid adjacency, then drawn with the
#                          paper's line styles (dotted / solid / dash-dot).
#
# =============================================================================


# All parameters closed over from the global nd_params dict
# (rho, eta, mu, alpha, beta are fixed; sigma and delta are the scan axes)


# Grid scan
# n_pts=150 → ~4-8 min.  Set to 200 for a smoother publication figure.
n_pts = 150
delta_vals = np.linspace(0.001, 1.5, n_pts)
sigma_vals = np.linspace(0.001, 1.3, n_pts)

region_map = np.zeros((n_pts, n_pts), dtype=int)

for i, d in enumerate(delta_vals):
    for j, s in enumerate(sigma_vals):
        r = classify_local(s, d)
        if r == 3:
            r = 5 if in_region_5(s, d) else 3
        region_map[j, i] = r

# Colors
color_by_region = {
    0: 'white',
    1: '#deebf7',   # pale blue, total regression only
    2: '#fee8d0',   # pale tan, single interior attractor
    3: '#e5f5e0',   # pale green, bistability (dormancy possible)
    4: '#fdae6b',   # amber, tiny region 4
    5: '#dadaeb',   # pale purple, sneaking-through / heteroclinic past
}
region_color_map = np.ones((n_pts, n_pts, 3))
for reg, color in color_by_region.items():
    region_color_map[region_map == reg] = mcolors.to_rgb(color)

fig4, ax4 = plt.subplots(figsize=(7, 6))
ax4.imshow(region_color_map, origin='lower',
           extent=[delta_vals[0], delta_vals[-1], sigma_vals[0], sigma_vals[-1]],
           aspect='auto', interpolation='nearest')

# Extract boundary curves from grid adjacency
def _boundary(map_arr, from_regs, to_regs, d_vals, s_vals, scan='right'):
    """Scan each sigma row for a from_regs → to_regs transition."""
    bd_d, bd_s = [], []
    for j, s in enumerate(s_vals):
        row = map_arr[j, :]
        rng = range(len(row)-1) if scan == 'right' else range(len(row)-2, -1, -1)
        for i in rng:
            if row[i] in from_regs and row[i+1] in to_regs:
                bd_d.append((d_vals[i] + d_vals[i+1]) / 2)
                bd_s.append(s)
                break
    arr_d, arr_s = np.array(bd_d), np.array(bd_s)
    idx = np.argsort(arr_s)
    return arr_d[idx], arr_s[idx]

# Saddle-node (upper): 2 → 3/5 going left-to-right
sn_up_d, sn_up_s = _boundary(region_map, {2}, {3,5}, delta_vals, sigma_vals)

# Saddle-node (lower): 3/5 → 2 going left-to-right
sn_lo_d, sn_lo_s = _boundary(region_map, {3,5}, {2}, delta_vals, sigma_vals)

# Heteroclinic: 5 → 3 going left-to-right
het_d, het_s = _boundary(region_map, {5}, {3}, delta_vals, sigma_vals)
# Smooth to reduce pixel-boundary noise
if len(het_d) > 8:
    het_d = gaussian_filter1d(het_d, sigma=3)

# Transcritical (analytical): σ = α·δ
delta_line = np.linspace(0.001, 1.5, 500)
tc_s_line  = nd_params['alpha'] * delta_line
tc_mask    = tc_s_line <= 1.35

# Draw curves
ax4.plot(delta_line[tc_mask], tc_s_line[tc_mask], 'k:',  lw=1.8)  # transcritical
ax4.plot(sn_up_d, sn_up_s,                        'k-',  lw=1.8)  # saddle-node upper
ax4.plot(sn_lo_d, sn_lo_s,                        'k-',  lw=1.8)  # saddle-node lower
if len(het_d) > 3:
    ax4.plot(het_d, het_s,                         'k-.', lw=1.8)  # heteroclinic

# Estimated parameter location
ax4.plot(nd_params['delta'], nd_params['sigma'], 'ko', ms=8, zorder=5)

# Region labels (positions tuned to match paper)
region_label_positions = {
    1: (0.12,  0.90),
    2: (1.00,  0.75),
    3: (0.65,  0.27),
    4: (0.045, 0.04),
    5: (0.65,  0.055),
}
for reg, (dx, sx) in region_label_positions.items():
    ax4.text(dx, sx, str(reg), fontsize=15, fontweight='bold', ha='center', va='center')

# Legend
region_descriptions = {
    1: 'Region 1: total regression',
    2: 'Region 2: single interior attractor',
    3: 'Region 3: bistability (dormancy)',
    4: 'Region 4: D + total regression',
    5: 'Region 5: sneaking-through',
}
patches = [mpatches.Patch(facecolor=color_by_region[r], edgecolor='grey',
                          label=region_descriptions[r])
           for r in [1,2,3,4,5]]
line_handles = [
    plt.Line2D([0],[0], color='k', ls=':',  lw=1.5, label='Transcritical bifurcation'),
    plt.Line2D([0],[0], color='k', ls='-',  lw=1.5, label='Saddle-node bifurcation'),
    plt.Line2D([0],[0], color='k', ls='-.', lw=1.5, label='Heteroclinic connection (A-C)'),
    plt.Line2D([0],[0], color='k', ls='',   marker='o', ms=6, label='Estimated parameters'),
]
ax4.legend(handles=patches + line_handles, loc='upper right',
           fontsize=7, framealpha=0.92)

ax4.set_xlabel(r'$\delta$ (effector death rate)', fontsize=12)
ax4.set_ylabel(r'$\sigma$ (effector source rate)', fontsize=12)
ax4.set_xlim(delta_vals[0], 1.45)
ax4.set_ylim(sigma_vals[0], 1.32)
ax4.set_title(
    'Figure 4: Bifurcation Diagram\n'
    r'($\delta$, $\sigma$ plane; all other parameters at estimated values)',
    fontsize=11
)
plt.tight_layout()
plt.show()

## Figure 5: Exploring the regions from Figure 4


The shared `PhasePortraitPlotter` class (imported from `phase_portrait.py`) does all the work here. We just instantiate one per (sigma, delta, mu) combination.


Now we can instantiate a plotter for each region and build figure 5

In [ ]:
# Figure 5 settings
# Format: sigma, delta, title, initial conditions, xlim, ylim, log_y
mu_base = nd_params['mu']

region_settings = [

    (
        0.318,
        0.1908,
        'Region 1',
        [
            (0.25,270),
            (0.55,55),
            (2.20,190),
        ],
        (-0.1,4.5),
        (-0.1,500),
        False
    ),

    (
        0.318,
        0.545,
        'Region 2',
        [
            (0.25,370),
            (3.00,270),
        ],
        (-0.1,4),
        (-0.1,500),
        False
    ),

    (
        0.182,
        0.545,
        'Region 3',
        [
            (0.25,100),
            (0.65,60),
            (1.20,490),
            (2.25,490),
        ],
        (-0.1,3),
        (-0.1,500),
        False
    ),

    (
        0.045,
        0.009,
        'Region 4',
        [
            (0.15,350),
            (0.35,130),
            (0.65,460),
            (0.55,60),
        ],
        (-0.1,5.2),
        (-0.1,500),
        False
    ),

    (
        0.073,
        0.545,
        'Region 5',
        [
            (3.3,110),
            (4.1,110),
            (0.15,460),
        ],
        (-0.1,5.2),
        (-0.1,500),
        False
    ),

    (
        0.073,
        0.545,
        'Region 5 (log y)',
        [
            (3.3,110),
            (4.1,110),
            (0.15,150),
        ],
        (-0.1,5.2),
        (0.3,1000),
        True
    ),
]


fig, axes = plt.subplots(
    3,
    2,
    figsize=(9,14)
)

axes = axes.flatten()

for ax, settings in zip(axes, region_settings):

    sigma, delta, title, initials, xlim, ylim, log_y = settings

    plotter = PhasePortraitPlotter.from_nd_params(nd_params, sigma=sigma, delta=delta, mu=mu_base)

    plotter.make_panel(
        ax,
        initials,
        title,
        xlim,
        ylim,
        log_y
    )

plt.suptitle(
    "Figure 5 Reconstruction",
    fontsize=14
)

plt.tight_layout()
plt.show()


## Figure 6: Moving around the crtitical mu value

In [ ]:
# Figure 6 fast schematic: basin shading from stable manifold of C
# Uses the shared PhasePortraitPlotter (stable_manifold_branches, plot_saddle_manifolds,
# plot_unstable_manifold_of_A, plot_fixed_points, label_fixed_points).

def fill_basin_from_stable_manifold(ax, branches, xlim=(-0.1, 3.0), ylim=(-0.1, 500),
                                    shade_left=True, fill_to_top=True):
    if len(branches) == 0:
        return

    branch = max(branches, key=lambda b: np.ptp(b[1]) if len(b[1]) > 1 else 0)
    x, y = branch

    if len(x) < 3:
        return

    idx = np.argsort(y)
    x, y = x[idx], y[idx]
    y_unique, unique_idx = np.unique(y, return_index=True)
    x_unique = x[unique_idx]

    if len(y_unique) < 3:
        return

    y0, y1 = max(ylim[0], y_unique.min()), min(ylim[1], y_unique.max())
    y_fill = np.geomspace(max(0.001, y0), y1, 400)
    x_boundary = np.interp(y_fill, y_unique, x_unique)

    if shade_left:
        ax.fill_betweenx(y_fill, xlim[0], x_boundary, color='lightgrey', alpha=0.75, zorder=0)
        if fill_to_top and y1 < ylim[1]:
            ax.fill_betweenx([y1, ylim[1]], xlim[0], x_boundary[-1], color='lightgrey', alpha=0.75, zorder=0)
    else:
        ax.fill_betweenx(y_fill, x_boundary, xlim[1], color='lightgrey', alpha=0.75, zorder=0)
        if fill_to_top and y1 < ylim[1]:
            ax.fill_betweenx([y1, ylim[1]], x_boundary[-1], xlim[1], color='lightgrey', alpha=0.75, zorder=0)


def make_figure6_panel_fast(ax, plotter, title, xlim=(-0.1, 3.0),
                            ylim=(-0.1, 500), shade_left=True):
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)

    # stable manifold used for fill
    branches = plotter.stable_manifold_branches(ax, T=60, max_step=5.0)

    # saddle manifolds
    plotter.plot_saddle_manifolds(ax, T_stable=60, T_unstable=60)

    # A unstable manifold
    plotter.plot_unstable_manifold_of_A(ax, T=60)

    plotter.plot_fixed_points(ax)
    plotter.label_fixed_points(ax)

    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_yscale('log')


# Run Figure 6

sigma_fig6 = 0.1181
delta_fig6 = 0.3743

fig6_settings = [
    (0.0048, "Figure 6a: before connection", False),
    (0.005014, "Figure 6b: near connection", False),
    (0.0055, "Figure 6c: after connection", True),
]

fig6, axes = plt.subplots(1, 3, figsize=(14, 4.5))

for ax, (mu_val, title, shade_left) in zip(axes, fig6_settings):
    plotter = PhasePortraitPlotter.from_nd_params(nd_params, sigma=sigma_fig6, delta=delta_fig6, mu=mu_val)
    make_figure6_panel_fast(ax, plotter, title,
                            xlim=(-0.1, 3.0), ylim=(0.001, 500), shade_left=shade_left)

plt.suptitle("Figure 6 Reconstruction: Heteroclinic basin transition")
plt.tight_layout()
plt.show()


## Figure 7: Time Series (Decaying Oscillations to Dormancy)

Initial conditions: y(0) = 50, x(0) = 5 (nondimensional).
Expect damped oscillations converging to steady state B over ~600 days.
Predicted oscillation period: 3-4 months.

In [ ]:
# TODO: integrate from (x0, y0) = (5, 50) to tau_max = 600 days

# timescale: tau = t_days * n * T0_nd
timescale = params['n'] * T0_nd
tau_max = 600 * timescale
t_evalF7 = np.linspace(0, tau_max, 50000)


sol7 = solve_ivp(model_nd, (0, tau_max), [5.0, 50.0],
                args=(sigma, rho, eta, mu, delta, alpha, beta), t_eval=t_evalF7,
                    dense_output=True)

t_days = sol7.t / timescale   # convert tau back to days



fig7, ax7 = plt.subplots(figsize=(6, 6))
ax7.semilogy(t_days, sol7.y[1], 'r--', lw=1.5, label='y (tumor)')
ax7.semilogy(t_days, sol7.y[0], 'b-',  lw=1.5, label='x (effector)')
ax7.set_ylim(1e-2, 1e2)
ax7.set_xlim(0, 600)
ax7.set_xlabel('Time (days)')
ax7.set_ylabel('x, y (dimensionless)')
ax7.legend()
ax7.set_title('Figure 7: Decaying Oscillations to Dormancy')
plt.tight_layout()
plt.show()

## Figure 8: Higher mu value


In [ ]:
# Figure 8 Reconstruction (presentation version)

mu_fig8 = 0.0063
sigma_fig8 = 0.1181
delta_fig8 = 0.3743

plotter_fig8 = PhasePortraitPlotter.from_nd_params(nd_params, sigma=sigma_fig8, delta=delta_fig8, mu=mu_fig8)

fig8_settings = [
    (
        'Figure 8a',
        [
            (0.4, 100),
            (5.5, 15),
        ],
        (-0.1, 6.0),
        (-0.1, 500),
        False
    ),
    (
        'Figure 8b (log y)',
        [
            (0.4, 100),
            (5.5, 15),
        ],
        (-0.1, 6.0),
        (1e-3, 1000),
        True
    ),
]

fig8, axes = plt.subplots(1, 2, figsize=(8, 4.5))

for ax, settings in zip(axes, fig8_settings):

    title, initials, xlim, ylim, log_y = settings

    # T_unstable extended so C's unstable manifold fully spirals into B,
    # matching the multi-loop spiral in the paper's figure. A sits at y=0,
    # so its own point is invisible on the log axis, but its unstable
    # manifold (y > 0 immediately off A) still plots fine.
    plotter_fig8.make_panel(ax, initials, title, xlim, ylim, log_y,
                            manifold_kwargs={'T_stable': 45, 'T_unstable': 250},
                            show_unstable_A=True)


plt.suptitle(r'Figure 8 Reconstruction: $\mu = 0.0055$', fontsize=14)
plt.tight_layout()
plt.show()


## Figure 9: Lower mu value


In [ ]:
# Figure 9 Reconstruction: mu = 0.0021

mu_fig9 = 0.0021

sigma_fig9 = 0.1181
delta_fig9 = 0.3743

plotter_fig9 = PhasePortraitPlotter.from_nd_params(nd_params, sigma=sigma_fig9, delta=delta_fig9, mu=mu_fig9)

fig9_settings = [
    (
        'Figure 9a',
        [
            (0.4, 100),
            (0.55, 450),
            (1.65, 490),
        ],
        (-0.1, 3.3),
        (-0.1, 500),
        False
    ),

    (
        'Figure 9b (log y)',
        [
            (0.4, 100),
            (0.55, 450),
            (1.65, 490),
        ],
        (-0.1, 3.3),
        (0.3, 1000),
        True
    ),
]


fig9, axes = plt.subplots(
    1,
    2,
    figsize=(9, 4.5)
)

for ax, settings in zip(axes, fig9_settings):

    title, initials, xlim, ylim, log_y = settings

    plotter_fig9.make_panel(
        ax,
        initials,
        title,
        xlim,
        ylim,
        log_y,
        show_unstable_A=True
    )


plt.suptitle(
    r"Figure 9 Reconstruction: $\mu = 0.0021$",
    fontsize=14
)

plt.tight_layout()
plt.show()
